Evaluating our Retrieval pipleine with Golden set of 25 questions 

In [1]:
%run rag_setup.py

c:\Users\sahil\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
D:\guar dai\ProofRAG\notebooks\rag_setup.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
W1122 14:01:46.048000 31888 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Setup ready. Chunks: 680


In [2]:
import json

with open("../evaluation/golden_set.json") as f:
    golden = json.load(f)

hits = 0
refused_right = 0
answerable = 0
unanswerable = 0

for item in golden:
    q = item["question"]
    results = vector_store.similarity_search_with_relevance_scores(q, k=3)
    best = results[0][1]
    files = [d.metadata["file"] for d, s in results]

    if item["expected_source"] is None:
        unanswerable += 1
        ok = best < 0.45
        refused_right += ok
        print(("PASS" if ok else "FAIL"), "| refuse check |", q, "| best:", round(best, 2))
    else:
        answerable += 1
        ok = item["expected_source"] in files
        hits += ok
        print(("PASS" if ok else "FAIL"), "| retrieval |", q)

print("\nRetrieval hit rate:", hits, "/", answerable)
print("Refusal accuracy:", refused_right, "/", unanswerable)


PASS | retrieval | How do I share a page with a guest?
PASS | retrieval | How can I change my Notion plan?
PASS | retrieval | Where can I see and update my billing information?
PASS | retrieval | What is Notion AI?
PASS | retrieval | How do I import a CSV file into Notion?
PASS | retrieval | How do I export my workspace content?
PASS | retrieval | What are rollups in a Notion database?
PASS | retrieval | How do database automations work?
PASS | retrieval | How can I set a reminder in Notion?
PASS | retrieval | How do I search for a page in Notion?
PASS | retrieval | What keyboard shortcuts are available in Notion?
PASS | retrieval | How do I publish a page as a website?
PASS | retrieval | How do comments and mentions work in Notion?
PASS | retrieval | What is a wiki in Notion?
PASS | retrieval | How do I create a form in Notion?
PASS | retrieval | How do I let someone outside my company see one of my pages?
PASS | retrieval | I want to pay for a better plan, where do I do that?
FAIL | 

C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\1375849238.py:13: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to 

In [3]:
def retrieve_final(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best = results[0][1]
    if best < 0.45:
        return best, []
    queries = make_queries(question)
    pool, seen = [], set()
    for q in queries:
        docs_q = vector_store.similarity_search(q, k=5) + bm25_retriever.invoke(q)[:5]
        for d in docs_q:
            if d.page_content not in seen:
                seen.add(d.page_content)
                pool.append(d)
    pairs = [(question, d.page_content) for d in pool]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(pool, scores), key=lambda x: x[1], reverse=True)
    return best, [d for d, s in ranked[:3]]


In [4]:
hits = 0
refused_right = 0
answerable = 0
unanswerable = 0

for item in golden:
    q = item["question"]
    best, docs = retrieve_final(q)
    files = [d.metadata["file"] for d in docs]

    if item["expected_source"] is None:
        unanswerable += 1
        ok = best < 0.45
        refused_right += ok
        print(("PASS" if ok else "FAIL"), "| refuse check |", q, "| best:", round(best, 2))
    else:
        answerable += 1
        ok = item["expected_source"] in files
        hits += ok
        print(("PASS" if ok else "FAIL"), "| pipeline |", q)

print("\nPipeline hit rate:", hits, "/", answerable)
print("Refusal accuracy:", refused_right, "/", unanswerable)


PASS | pipeline | How do I share a page with a guest?
PASS | pipeline | How can I change my Notion plan?
FAIL | pipeline | Where can I see and update my billing information?
PASS | pipeline | What is Notion AI?
PASS | pipeline | How do I import a CSV file into Notion?
PASS | pipeline | How do I export my workspace content?
FAIL | pipeline | What are rollups in a Notion database?
PASS | pipeline | How do database automations work?
PASS | pipeline | How can I set a reminder in Notion?
PASS | pipeline | How do I search for a page in Notion?
PASS | pipeline | What keyboard shortcuts are available in Notion?
FAIL | pipeline | How do I publish a page as a website?
PASS | pipeline | How do comments and mentions work in Notion?
FAIL | pipeline | What is a wiki in Notion?
PASS | pipeline | How do I create a form in Notion?
PASS | pipeline | How do I let someone outside my company see one of my pages?
FAIL | pipeline | I want to pay for a better plan, where do I do that?
PASS | pipeline | Can I 

C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\3058651645.py:2: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n

Our pipeline is behaving poorly than the baseline model


Let's diagnose 


In [6]:
q = "What is a wiki in Notion?"

print("== Semantic top 3 (baseline) ==")
for d, s in vector_store.similarity_search_with_relevance_scores(q, k=3):
    print(round(s, 2), d.metadata["file"])

qs = make_queries(q)
pool = pool_for(qs)
print("\nPool size:", len(pool))
print("Is wiki chunk in pool ?", any(d.metadata["file"] == "wikis.md" for d in pool))

pairs = [(q, d.page_content) for d in pool]
scores = reranker.predict(pairs)
ranked = sorted(zip(pool, scores), key=lambda x: x[1], reverse=True)

print("\n== top 5 after rerank ==")
for d, s in ranked[:5]:
    print(round(float(s), 2), d.metadata["file"])

for i, (d, s) in enumerate(ranked):
    if d.metadata["file"] == "wikis.md":
        print("\nWiki  best chunk rerank position:", i + 1, "| score:", round(float(s), 2))
        break


== Semantic top 3 (baseline) ==
0.47 writing-and-editing-basics.md
0.41 wikis.md
0.37 wikis.md

Pool size: 22
Is wiki chunk in pool ? True

== top 5 after rerank ==
1.51 search-in-notion.md
1.37 writing-and-editing-basics.md
0.69 intro-to-workspaces (1).md
-1.28 notion-sites-publishing.md
-1.97 notion-ai-faqs.md

Wiki  best chunk rerank position: 7 | score: -5.4


In [7]:
def retrieve_rerank_only(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best = results[0][1]
    if best < 0.45:
        return best, []
    docs10 = [d for d, s in results]
    pairs = [(question, d.page_content) for d in docs10]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs10, scores), key=lambda x: x[1], reverse=True)
    return best, [d for d, s in ranked[:3]]

def retrieve_hybrid_no_mq(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best = results[0][1]
    if best < 0.45:
        return best, []
    pool, seen = [], set()
    for d in [doc for doc, s in results] + bm25_retriever.invoke(question):
        if d.page_content not in seen:
            seen.add(d.page_content)
            pool.append(d)
    pairs = [(question, d.page_content) for d in pool]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(pool, scores), key=lambda x: x[1], reverse=True)
    return best, [d for d, s in ranked[:3]]

def run_eval(retrieve_fn, label):
    hits = refused = ans = unans = 0
    fails = []
    for item in golden:
        q = item["question"]
        best, docs = retrieve_fn(q)
        files = [d.metadata["file"] for d in docs]
        if item["expected_source"] is None:
            unans += 1
            ok = best < 0.45
            refused += ok
        else:
            ans += 1
            ok = item["expected_source"] in files
            hits += ok
        if not ok:
            fails.append(q)
    print(label, "| hit rate:", hits, "/", ans, "| refusal:", refused, "/", unans)
    for f in fails:
        print("   FAIL:", f)

run_eval(retrieve_rerank_only, "B: semantic + rerank")
run_eval(retrieve_hybrid_no_mq, "C: hybrid, no multi-query")


C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\1423246560.py:2: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n

B: semantic + rerank | hit rate: 15 / 20 | refusal: 5 / 5
   FAIL: Where can I see and update my billing information?
   FAIL: What are rollups in a Notion database?
   FAIL: How do I publish a page as a website?
   FAIL: What is a wiki in Notion?
   FAIL: I want to pay for a better plan, where do I do that?
C: hybrid, no multi-query | hit rate: 15 / 20 | refusal: 5 / 5
   FAIL: Where can I see and update my billing information?
   FAIL: What are rollups in a Notion database?
   FAIL: How do I publish a page as a website?
   FAIL: What is a wiki in Notion?
   FAIL: I want to pay for a better plan, where do I do that?


C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\1423246560.py:13: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to 

In [8]:
reranker_big = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")

def retrieve_rerank_model(question, model):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best = results[0][1]
    if best < 0.45:
        return best, []
    docs10 = [d for d, s in results]
    pairs = [(question, d.page_content) for d in docs10]
    scores = model.predict(pairs)
    ranked = sorted(zip(docs10, scores), key=lambda x: x[1], reverse=True)
    return best, [d for d, s in ranked[:3]]

run_eval(lambda q: retrieve_rerank_model(q, reranker_big), "E: semantic + rerank L-12")


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

c:\Users\sahil\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sahil\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\sahil\AppData\Local\Programs\Python\Python313\Lib\site-packages\hug

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

E: semantic + rerank L-12 | hit rate: 15 / 20 | refusal: 5 / 5
   FAIL: Where can I see and update my billing information?
   FAIL: What are rollups in a Notion database?
   FAIL: How do I publish a page as a website?
   FAIL: What is a wiki in Notion?
   FAIL: I want to pay for a better plan, where do I do that?


C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\1407746862.py:4: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n

In [9]:
def retrieve_rrf(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=10)
    best = results[0][1]
    if best < 0.45:
        return best, []
    sem_docs = [d for d, s in results]
    bm25_docs = bm25_retriever.invoke(question)
    rrf = {}
    docs_by_text = {}
    for rank, d in enumerate(sem_docs):
        rrf[d.page_content] = rrf.get(d.page_content, 0) + 1 / (60 + rank + 1)
        docs_by_text[d.page_content] = d
    for rank, d in enumerate(bm25_docs):
        rrf[d.page_content] = rrf.get(d.page_content, 0) + 1 / (60 + rank + 1)
        docs_by_text[d.page_content] = d
    top = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:3]
    return best, [docs_by_text[t] for t, s in top]

run_eval(retrieve_rrf, "F: hybrid + RRF, no reranker")


F: hybrid + RRF, no reranker | hit rate: 15 / 20 | refusal: 5 / 5
   FAIL: Where can I see and update my billing information?
   FAIL: What are rollups in a Notion database?
   FAIL: How do I publish a page as a website?
   FAIL: I want to pay for a better plan, where do I do that?
   FAIL: Can I bring my data from another app into Notion?


C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\1592412260.py:2: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n

Both fixes failed too. Here's the full scoreboard now:

A — semantic only: 19/20 👑
B — semantic + rerank (L-6): 15/20
C — hybrid + rerank: 15/20
D — full pipeline: 15/20
E — semantic + rerank (L-12, bigger): 15/20
F — hybrid + RRF, no reranker: 15/20

In [ ]:
q = "What are rollups in a Notion database?"
best, docs = retrieve_rerank_only(q)
for d in docs:
    print("----", d.metadata["file"])
    print(d.page_content[:300])




In [15]:
print("----", d.metadata["file"])
print(d.page_content[:300])


---- wikis.md
- All pages: This is a database view of all the pages within the wiki.
- Pages I own: This is a database view showing only the pages that you own.


In [16]:
def answer_with_semantic(question):
    results = vector_store.similarity_search_with_relevance_scores(question, k=3)
    best = results[0][1]
    if best < 0.45:
        return "REFUSED", best
    docs = [d for d, s in results]
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: I don't have this information in the documents.

Context:
{context}

Question: {question}

Answer:"""
    return llm.invoke(prompt).content, best

for item in golden:
    ans, score = answer_with_semantic(item["question"])
    print("=" * 60)
    print("Q:", item["question"], "| expected:", item["expected_source"], "| best:", round(score, 2))
    print("A:", ans[:400])


Q: How do I share a page with a guest? | expected: sharing-and-permissions.md | best: 0.67
A: 1. Click **Share** at the top of the page.  
2. In the sharing dialog, enter the guest’s email address (or select their name if they’re already in the workspace).  
3. Click the dropdown next to their name and choose the desired permission level.  
4. Press **Invite**.  

The guest will receive an invitation and, once they accept, will have the permissions you set. (If the email belongs to one of 
Q: How can I change my Notion plan? | expected: change-your-plan.md | best: 0.66
A: To change your Notion plan:

1. In the sidebar, go to **Settings → Billing → Change plan**.  
2. Choose the plan you want to move to.  
3. Click **Continue**.  
4. If you’re downgrading, provide any requested feedback and then click **Downgrade**.  
5. You’ll receive a confirmation and see when your current plan ends.  
Q: Where can I see and update my billing information? | expected: billing.md | best: 0.37
A: REFUSE

C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\3221757775.py:2: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n

In [18]:
import json
golden = json.load(open("../evaluation/golden_set_v2.json"))
print("Questions:", len(golden))

hits = refused = ans = unans = 0
fails = []
for item in golden:
    q = item["question"]
    results = vector_store.similarity_search_with_relevance_scores(q, k=3)
    best = results[0][1]
    files = [d.metadata["file"] for d, s in results]
    if item["expected_source"] is None:
        unans += 1
        ok = best < 0.45
        refused += ok
    else:
        ans += 1
        ok = item["expected_source"] in files
        hits += ok
    if not ok:
        fails.append((q, item["expected_source"], round(best, 2)))

print("Hit rate:", hits, "/", ans)
print("Refusal accuracy:", refused, "/", unans)
for f in fails:
    print("FAIL:", f)


Questions: 100


C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\443299672.py:9: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to na

Hit rate: 79 / 85
Refusal accuracy: 10 / 15
FAIL: ('Can I bring my data from another app into Notion?', 'import-data.md', np.float32(0.47))
FAIL: ('What sections does the Notion sidebar contain?', 'intro-to-workspaces.md', np.float32(0.53))
FAIL: ('How do I add a page to my Favorites in the sidebar?', 'intro-to-workspaces.md', np.float32(0.61))
FAIL: ('Can Notion assign a task to someone automatically when its status changes?', 'database-automations.md', np.float32(0.33))
FAIL: ("We have an intern who only needs to see two specific pages and nothing else in the workspace. What's the best way to add her?", 'whos-who-in-a-workspace.md', np.float32(0.28))
FAIL: ('How do I stop anyone from changing a really important page by accident?', 'intro-to-workspaces.md', np.float32(0.38))
FAIL: ('How much revenue did Notion earn last year?', None, np.float32(0.45))
FAIL: ('What is the exact monthly price of the Notion Plus plan per user?', None, np.float32(0.56))
FAIL: ('Can I pay for my Notion pla

In [19]:
import time

lines = []
gate_refused = llm_refused = answered = 0
for item in golden:
    ans, score = answer_with_semantic(item["question"])
    if ans == "REFUSED":
        gate_refused += 1
        tag = "GATE-REFUSED"
    elif ans.startswith("I don't have this information"):
        llm_refused += 1
        tag = "LLM-REFUSED"
    else:
        answered += 1
        tag = "ANSWERED"
    lines.append(f"[{tag}] Q: {item['question']} | expected: {item['expected_source']} | best: {round(score, 2)}\nA: {ans}\n")
    time.sleep(2)

open("../evaluation/answer_eval_v2.txt", "w", encoding="utf-8").write("\n".join(lines))
print("Gate refused:", gate_refused, "| LLM refused:", llm_refused, "| Answered:", answered)


C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\3221757775.py:2: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n

Gate refused: 30 | LLM refused: 12 | Answered: 58


In [20]:
def reformulate(question, attempt):
    styles = {
        1: "Rewrite this question using formal words that would appear in help documentation.",
        2: "Rewrite this question in the simplest possible words, as a beginner would ask it.",
    }
    prompt = f"""{styles[attempt]}
Write only the rewritten question, nothing else.

Question: {question}"""
    return llm.invoke(prompt).content.strip()

def answer_with_retry(question, max_retries=2):
    ans, score = answer_with_semantic(question)
    if ans != "REFUSED" and not ans.startswith("I don't have this information"):
        return ans, score, 0
    for attempt in range(1, max_retries + 1):
        new_q = reformulate(question, attempt)
        ans, score = answer_with_semantic(new_q)
        if ans != "REFUSED" and not ans.startswith("I don't have this information"):
            return ans, score, attempt
    return "REFUSED", score, max_retries


In [21]:
recovered = still_refused = fine_first_try = danger = 0
for item in golden:
    ans, score, attempts = answer_with_retry(item["question"])
    refused = ans == "REFUSED" or ans.startswith("I don't have this information")
    if item["expected_source"] is None:
        if not refused:
            danger += 1
            print("DANGER:", item["question"], "->", ans[:100])
    else:
        if refused:
            still_refused += 1
        elif attempts > 0:
            recovered += 1
            print("RECOVERED (attempt", attempts, "):", item["question"])
        else:
            fine_first_try += 1
    time.sleep(1)

print("Fine first try:", fine_first_try)
print("Recovered by retry:", recovered)
print("Still refused:", still_refused)
print("Dangerous answers:", danger)


RECOVERED (attempt 1 ): I want to pay for a better plan, where do I do that?
RECOVERED (attempt 1 ): Can I bring my data from another app into Notion?


C:\Users\sahil\AppData\Local\Temp\ipykernel_31888\3221757775.py:2: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8ff689da-4e51-4e24-b654-d0a4bc5e1381', metadata={'title': 'Database properties in Notion', 'source': 'https://www.notion.com/en-gb/help/database-properties', 'file': 'database-properties.md'}, page_content='- Conditional colour is available on the following views:\n  - Table\n  - Calendar\n  - Timeline\n  - List\n  - Board\n  - Feed\n- Conditional colour can be applied based on the following properties:\n  - Select\n  - Multi-select\n  - Status\n  - Title\n  - Text\n  - Number\n  - Date\n  - Person\n  - Tick box\n  - Formulas\n  - Relations\n  - Rollups'), np.float32(-0.12533307)), (Document(id='aa600e16-31da-40f6-8eac-b262d2b43a02', metadata={'title': 'Getting started with Notion Calendar', 'source': 'https://www.notion.com/help/guides/getting-started-with-notion-calendar', 'file': 'notion-calendar.md'}, page_content='- Go to today — Press T to n

RECOVERED (attempt 1 ): What are the Notion AI image generation limits per user?
RECOVERED (attempt 1 ): How do I turn an existing page into a wiki?
RECOVERED (attempt 2 ): How can I share a form so that people outside my workspace can fill it out?
RECOVERED (attempt 1 ): How do I let colleagues book a meeting with me through Notion Calendar?
RECOVERED (attempt 1 ): All my notes are in Word files on my laptop. Can I move them into Notion?
DANGER: How do I connect Notion with Microsoft Teams? -> You can’t connect Notion to Microsoft Teams. Members in this role don’t see the Connect option in Se
DANGER: Which AI model does Notion AI use to answer questions? -> Notion uses **Notion AI**, which can draw on the pages in your workspace, connected apps, web inform
Fine first try: 57
Recovered by retry: 7
Still refused: 21
Dangerous answers: 2
